<a href="https://colab.research.google.com/github/kihahu/kikuyu-tts/blob/initial-import/notebooks/train_kikuyu_vits_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Kikuyu TTS Colab Workflow

This notebook now prioritizes the MMS-first workflow for Kikuyu-capable open-weight TTS.

Default flow:
1. Set up the Colab runtime and mount Drive.
2. Prepare Waxal manifests, including the canonical single-speaker split.
3. Run the MMS candidate-selection/export workflow.
4. Inspect the selected base and synthesize a quick test sample.

Secondary flow:
- Scratch Coqui VITS training remains available later in the notebook if you need a fully trainable in-repo baseline.


In [ ]:
!pip install -U pip setuptools wheel
!pip install 'datasets[audio]' soundfile librosa pyyaml huggingface_hub transformers accelerate sentencepiece safetensors
!pip install coqui-tts


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!cd /content && ([ -d kikuyu-tts ] || git clone https://github.com/kihahu/kikuyu-tts.git) && cd kikuyu-tts && git pull
%cd /content/kikuyu-tts


In [ ]:
!python scripts/prepare_waxal_kik_tts.py \
  --dataset-name google/WaxalNLP \
  --dataset-config kik_tts \
  --split train \
  --output-dir data/waxal_kik_tts \
  --target-sample-rate 16000 \
  --min-duration-sec 0.6 \
  --max-duration-sec 25.0 \
  --min-rms 0.0035 \
  --seed 42 \
  --dev-ratio 0.10 \
  --test-ratio 0.05


## Preferred Path: MMS Base Selection And Export

This chooses among the ranked MMS shortlist, benchmarks the candidates on fixed Kikuyu prompts, and exports the selected base model in Hugging Face format under `artifacts/mms_tts_selected_base`.


In [ ]:
!python scripts/prepare_mms_tts_finetune.py \
  --config configs/finetune_mms_tts_kik.yaml


In [ ]:
import json
from pathlib import Path

report_path = Path("artifacts/mms_tts_selection_report.json")
with report_path.open("r", encoding="utf-8") as f:
    report = json.load(f)

print("Resolved base model:", report["resolved_base_model"])
print("Selection strategy:", report["selection_strategy"])
print("Canonical speaker:", report["canonical_single_speaker"])
for item in report["candidate_reports"]:
    print(item)


In [ ]:
from pathlib import Path
import torch
from IPython.display import Audio, display
from transformers import AutoTokenizer, VitsModel

model_dir = Path("artifacts/mms_tts_selected_base")
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = VitsModel.from_pretrained(model_dir)

text = "Nĩ ndaugire nĩ wega mũno kũmenya ũhoro ũcio."
inputs = tokenizer(text, return_tensors="pt")
with torch.inference_mode():
    output = model(**inputs)
wav = output.waveform.squeeze().detach().cpu().numpy()

sample_path = "/content/mms_assessment_sample.wav"
import soundfile as sf
sf.write(sample_path, wav, model.config.sampling_rate)
display(Audio(sample_path, rate=model.config.sampling_rate, autoplay=True))


## Secondary Path: Scratch Coqui VITS Training

Use this only if you need a fully trainable in-repo baseline today. This path takes longer and is riskier than the MMS-first export workflow above.


In [ ]:
!python scripts/build_kikuyu_vocab.py \
  --train-manifest data/waxal_kik_tts/manifests/train.jsonl \
  --dev-manifest data/waxal_kik_tts/manifests/dev.jsonl \
  --out-dir artifacts/tokenizer_kikuyu_char \
  --wikipedia-orthography \
  --extra-chars 'ñÑ'


In [ ]:
# Optional: if you have broken `TTS`/`trainer` wheel state, nuke the cache and reinstall.
# !pip uninstall -y TTS trainer coqpit
# !pip cache purge
# !pip install -U coqui-tts
# !pip install -e .
# %cd /content/kikuyu-tts


In [ ]:
# Start scratch Coqui VITS training (writes coqui_vits_config.yaml, then runs train_tts)
!python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/kikuyu-tts


In [ ]:
# Resume from latest checkpoint on Drive (see local_output_path / config)
!python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/kikuyu-tts \
  --resume


In [ ]:
# Optional: push checkpoints to Hugging Face Hub
!huggingface-cli login
!python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/kikuyu-tts \
  --resume \
  --push-hf


Create a metrics CSV at `artifacts/checkpoint_metrics.csv` with columns:
- `checkpoint`
- `synthesis_success_rate`
- `clipping_rate`
- `mos_lite`
- `wer_proxy`


In [ ]:
!python scripts/evaluate_and_select.py \
  --metrics-csv artifacts/checkpoint_metrics.csv \
  --out-json artifacts/best_checkpoint_selection.json \
  --out-csv artifacts/tts_eval_summary.csv


In [ ]:
!python scripts/prepare_local_integration.py \
  --best-checkpoint-dir artifacts/colab_runs/kikuyu_vits_scratch/checkpoint_best \
  --tokenizer-dir artifacts/tokenizer_kikuyu_char \
  --eval-summary-csv artifacts/tts_eval_summary.csv \
  --out-dir artifacts/local_integration/kikuyu_vits_best \
  --model-id kikuyu-vits-scratch-waxal
